[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Learning Rate and Trees &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' data objects and curve functions, rebuilt here from the same made-up stations so
that each answer runs on its own.

In [2]:
y_train, y_valid, y_test = train["tmin"], valid["tmin"], test["tmin"]
train_dataset = lgb.Dataset(train[FEATURES], label=y_train, free_raw_data=False)
valid_dataset = lgb.Dataset(valid[FEATURES], label=y_valid, reference=train_dataset)
dtrain = xgb.DMatrix(train[FEATURES], label=y_train)
dvalid = xgb.DMatrix(valid[FEATURES], label=y_valid)
dtest = xgb.DMatrix(test[FEATURES], label=y_test)


def lightgbm_curve(rate, rounds=2000):
    losses = {}
    booster = lgb.train({"objective": "regression", "learning_rate": rate, "seed": 0, "verbose": -1},
                        train_dataset, rounds, valid_sets=[valid_dataset], valid_names=["valid"],
                        callbacks=[lgb.record_evaluation(losses)])
    return booster, np.array(losses["valid"]["l2"])


def xgboost_curve(rate, rounds=2000):
    losses = {}
    booster = xgb.train({"objective": "reg:squarederror", "learning_rate": rate, "seed": 0}, dtrain, rounds,
                        evals=[(dvalid, "valid")], evals_result=losses, verbose_eval=False)
    return booster, np.array(losses["valid"]["rmse"]) ** 2

**1.** Record LightGBM's validation loss at a learning rate of 0.05 for 2,000 rounds with `lightgbm_curve`, and print its best round and best loss.

In [3]:
booster, curve = lightgbm_curve(0.05)
print("best round", int(curve.argmin()) + 1, "| best loss", round(curve.min(), 3))

best round 361 | best loss 2.48


At 0.05 LightGBM's validation loss is lowest after 361 rounds, at 2.48, between 0.1's 120 rounds and 0.03's 1,130, and between their best losses too.

**2.** Do the same for XGBoost with `xgboost_curve` at 0.05.

In [4]:
booster, curve = xgboost_curve(0.05)
print("best round", int(curve.argmin()) + 1, "| best loss", round(curve.min(), 3))

best round 492 | best loss 2.515


XGBoost at 0.05 is best after 492 rounds, at 2.515, more rounds than its best at 0.03 took, 389. Near their lowest points these curves are so flat that a small change in the loss moves the best round a long way.

**3.** Take the LightGBM booster at a learning rate of 0.03 from `lightgbm_boosters`, and print its validation loss from its first 50, 500 and 1,500 trees.

In [5]:
booster, curve = lightgbm_curve(0.03)
for trees in (50, 500, 1500):
    print(trees, "trees:", round(mean_squared_error(y_valid, booster.predict(valid[FEATURES], num_iteration=trees)), 3))

50 trees: 6.778
500 trees: 2.438
1500 trees: 2.457


The solutions notebook has no `lightgbm_boosters`, so the cell trains the same booster again with `lightgbm_curve(0.03)`. The booster at 0.03 is far from done after 50 trees, at 6.778, close to its best after 500, at 2.438, and slightly worse after 1,500, at 2.457, past its best round of 1,130.

**4.** Train an XGBoost booster for 100 rounds at 0.1, then give it 200 more with `xgb_model`, and print the validation loss after 100 and after 300 rounds.

In [6]:
params = {"objective": "reg:squarederror", "learning_rate": 0.1, "seed": 0}
first = xgb.train(params, dtrain, 100)
more = xgb.train(params, dtrain, 200, xgb_model=first)
print(first.num_boosted_rounds(), "rounds:", round(mean_squared_error(y_valid, first.predict(dvalid)), 3))
print(more.num_boosted_rounds(), "rounds:", round(mean_squared_error(y_valid, more.predict(dvalid)), 3))

100 rounds: 2.544
300 rounds: 2.531


The 200 added rounds take the validation loss from 2.544 to 2.531, which is what the recorded curve at 0.1 shows for round 300.

**5.** Fit an `XGBRegressor` with the learning rate XGBoost chose in the worked examples and `n_estimators` set to its best round, and print its test loss beside the booster's.

In [7]:
booster, curve = xgboost_curve(0.03)
best = int(curve.argmin()) + 1
model = xgb.XGBRegressor(learning_rate=0.03, n_estimators=best, random_state=0).fit(train[FEATURES], y_train)
print("XGBRegressor:", round(mean_squared_error(y_test, model.predict(test[FEATURES])), 3))
print("booster:", round(mean_squared_error(y_test, booster.predict(dtest, iteration_range=(0, best))), 3))

XGBRegressor: 2.53
booster: 2.53


The scikit-learn model with the same rate and the same 389 trees gives the same test loss as the booster's first 389 trees, 2.53.

---

&#8592; **Back to:** [Learning Rate and Trees](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/04-learning-rate-and-trees.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
